# 🏉 Bradford Bulls - Logo Exposure Detection (Phase 1: MVP)


> **CẢNH BÁO MÔI TRƯỜNG PRODUCTION**: Notebook này chạy xử lý thực tế trên dữ liệu video (Không dùng dummy data). Module nhận diện Logo ở MVP này sử dụng SIFT Feature Matching để không cần gán nhãn (labeling) hàng ngàn ảnh cho CNN ngay lập tức. Tính năng này sẽ chứng minh pipeline hoạt động đầu cuối.


## 1. Setup Environment & Drive Mount


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# CẤU HÌNH ĐƯỜNG DẪN CỦA DỰ ÁN TRÊN GOOGLE DRIVE
PROJECT_ROOT = '/content/drive/MyDrive/Bradford_Bulls_AI'
VIDEO_DIR = os.path.join(PROJECT_ROOT, 'videos')
SPONSOR_DIR = os.path.join(PROJECT_ROOT, 'Sponsor Logo')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Đã kết nối thư mục làm việc: {PROJECT_ROOT}")


In [ ]:
!pip install ultralytics opencv-python-headless
!pip install lapx # Cần thiết cho ByteTrack Object Tracking


## 2. Core Config & Pipeline Initialization


In [ ]:
import cv2
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict
from ultralytics import YOLO

# Cấu hình cụ thể cho Phase 1: Chỉ tập trung chứng minh Main Chest Sponsor
SPONSOR_CONFIG = {
    "topnotch": {
        "name": "Top Notch", 
        "kit": "home", 
        "logo_files": ["13 - Top Notch Logo.png"]
    },
    "floor_tonic": {
        "name": "Floor Tonic", 
        "kit": "away", 
        "logo_files": ["Floor tonic Logo.jpg"]
    }
}

# Thông số hệ thống
CONF_PLAYER = 0.50
SAMPLE_RATE = 2  # Xử lý 2 frames/giây (Tối ưu MVP để ko chạy mất quá nhiều giờ)
SIFT_MATCH_THRESHOLD = 8  # Cần ít nhất 8 điểm tương đồng (good matches) để coi là nhận diện logo thành công


## 3. Pose ROI & SIFT Logo Matcher (Real Algorithm)


In [ ]:
class PoseROIExtractor:
    def __init__(self):
        self.L_SHOULDER, self.R_SHOULDER = 5, 6
        self.L_HIP, self.R_HIP = 11, 12
        
    def get_chest_roi(self, keypoints, bbox_w, bbox_h):
        """Tính toạ độ vùng ngực (Chest) thực tế dựa trên điểm khớp"""
        kpts = keypoints # (17, 3) -> x, y, conf
        
        # Kiểm tra độ tin cậy của khung xương vai và hông
        if (kpts[self.L_SHOULDER][2] > 0.4 and kpts[self.R_SHOULDER][2] > 0.4 and 
            kpts[self.L_HIP][2] > 0.4 and kpts[self.R_HIP][2] > 0.4):
            
            x1 = min(kpts[self.L_SHOULDER][0], kpts[self.L_HIP][0])
            x2 = max(kpts[self.R_SHOULDER][0], kpts[self.R_HIP][0])
            y1 = min(kpts[self.L_SHOULDER][1], kpts[self.R_SHOULDER][1])
            y2 = max(kpts[self.L_HIP][1], kpts[self.R_HIP][1])
            
            # Thêm margin an toàn (10%)
            margin_x = (x2 - x1) * 0.1
            margin_y = (y2 - y1) * 0.1
            
            x1 = max(0, int(x1 - margin_x))
            y1 = max(0, int(y1 - margin_y))
            x2 = min(bbox_w, int(x2 + margin_x))
            y2 = min(bbox_h, int(y2 + margin_y))
            
            if (x2 - x1) > 20 and (y2 - y1) > 20: # Region đủ lớn
                return [x1, y1, x2, y2]
        return None

class RealLogoRecognizer:
    """Sử dụng SIFT Feature Matching để nhận diện logo không cần training CNN"""
    def __init__(self, templates_dir, config):
        self.sift = cv2.SIFT_create()
        self.templates = {}
        
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        self.flann = cv2.FlannBasedMatcher(index_params, search_params)
        
        # Load real image templates
        print("Tải Logo Templates...")
        for sp_key, sp_data in config.items():
            self.templates[sp_key] = []
            for filename in sp_data['logo_files']:
                path = os.path.join(templates_dir, filename)
                if os.path.exists(path):
                    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        # Resize lớn để trích xuất được nhiều đặc trưng SIFT
                        img = cv2.resize(img, (300, int(300 * img.shape[0] / img.shape[1])))
                        kp, des = self.sift.detectAndCompute(img, None)
                        if des is not None:
                            self.templates[sp_key].append({'img': img, 'des': des})
                            print(f"✅ Loaded {filename} - SIFT Features: {len(kp)}")
                else:
                    print(f"❌ Missing file: {path}")

    def detect_chest_logo(self, chest_crop, kit_type):
        if chest_crop is None or chest_crop.size == 0: return None
        
        gray_crop = cv2.cvtColor(chest_crop, cv2.COLOR_BGR2GRAY)
        kp_roi, des_roi = self.sift.detectAndCompute(gray_crop, None)
        
        if des_roi is None or len(des_roi) < 5: return None
        
        target_sponsors = ['topnotch'] if kit_type == 'home' else ['floor_tonic']
        
        best_match_count = 0
        best_sponsor = None
        
        for sp_key in target_sponsors:
            for template in self.templates.get(sp_key, []):
                des_template = template['des']
                if des_template is None or len(des_template) < 2: continue
                
                try:
                    matches = self.flann.knnMatch(des_template, des_roi, k=2)
                    # Lowe's ratio test (Lọc nhiễu)
                    good_matches = [m for m, n in matches if m.distance < 0.7 * n.distance]
                    
                    if len(good_matches) > best_match_count:
                        best_match_count = len(good_matches)
                        best_sponsor = sp_key
                except: pass
                
        if best_match_count >= SIFT_MATCH_THRESHOLD:
            return best_sponsor
            
        return None


## 4. Thực thi Core MVP Pipeline


In [ ]:
def process_mvp_match(video_name, kit_type):
    video_path = os.path.join(VIDEO_DIR, video_name)
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Không tìm thấy video: {video_path}")
        
    print(f"\n--- KHỞI ĐỘNG PIPELINE PHASE 1 ---")
    print(f"Video: {video_name} | Kit: {kit_type}")
    
    # Init modules
    pose_model = YOLO("yolov8n-pose.pt") # Tải weights Pose
    roi_extractor = PoseROIExtractor()
    logo_recognizer = RealLogoRecognizer(SPONSOR_DIR, SPONSOR_CONFIG)
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30.0
    frame_interval = int(fps / SAMPLE_RATE)
    
    # Theo dõi Data thực
    raw_exposure_seconds = defaultdict(float)
    
    frame_count = 0
    processed_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video FPS: {fps} | Tổng Frames: {total_frames} | Khoảng cách lấy mẫu: {frame_interval} frames")
    
    # Trong môi trường Colab, dùng tracking module để track objects qua các frame
    # YOLO track sử dụng ByteTrack (yêu cầu lapx)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        frame_count += 1
        
        # Rút ngắn thời gian xử lý: Chỉ xử lý 1 phần nhỏ video để test nếu muốn
        # Ở production, để chạy hết thì xoá/comment đoạn giới hạn này.
        # if frame_count > 5000: break 
        
        if frame_count % frame_interval != 0: continue
            
        processed_count += 1
        
        # 1. Player Tracking & Pose Detection (Real Process)
        # Sử dụng tracker 'bytetrack.yaml' tích hợp sẵn của YOLOv8
        results = pose_model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=CONF_PLAYER)
        
        for r in results:
            if r.boxes.id is None: continue # Bỏ qua nếu không track được ID
            
            boxes = r.boxes.xyxy.cpu().numpy()
            track_ids = r.boxes.id.int().cpu().tolist()
            keypoints = r.keypoints.data.cpu().numpy() 
            
            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box)
                kpts = keypoints[i]
                
                # 2. Lấy ROI Vùng Ngực
                # Chuyển toạ độ kpts về toạ độ tương đối trong khung hình bounding box
                rel_kpts = kpts.copy()
                rel_kpts[:, 0] -= x1
                rel_kpts[:, 1] -= y1
                
                chest_roi = roi_extractor.get_chest_roi(rel_kpts, x2-x1, y2-y1)
                
                if chest_roi:
                    cx1, cy1, cx2, cy2 = chest_roi
                    chest_crop = frame[y1+cy1 : y1+cy2, x1+cx1 : x1+cx2]
                    
                    # 3. Chạy thuật toán Logo Matcher (Thật)
                    detected_sponsor = logo_recognizer.detect_chest_logo(chest_crop, kit_type)
                    
                    if detected_sponsor:
                        # Cộng thời gian thật: 1 frame đại diện cho 1/SAMPLE_RATE giây
                        raw_exposure_seconds[detected_sponsor] += (1.0 / SAMPLE_RATE)
        
        if processed_count % 100 == 0:
            print(f"Đã xử lý {processed_count} sampled frames ({frame_count}/{total_frames})...")
            
    cap.release()
    print("\n=== KẾT QUẢ THỰC TẾ ===")
    for sponsor, sec in raw_exposure_seconds.items():
        print(f"👉 {SPONSOR_CONFIG[sponsor]['name']}: {sec:.2f} giây")
        
    return raw_exposure_seconds



## 5. Kích Hoạt Xử Lý
Chạy thử thách MVP trên video thực tế sân khách (M06).


In [ ]:
# Xử lý video M06 (Sân khách - Áo đen, Logo ngực là Floor Tonic)
try:
    real_results = process_mvp_match("M06_black_1080p.mp4", "away")
    
    if real_results:
        # Xuất CSV thực tế
        df = pd.DataFrame([
            {"Sponsor": SPONSOR_CONFIG[k]["name"], "Raw Exposure (Seconds)": v} 
            for k, v in real_results.items()
        ])
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(OUTPUT_DIR, f"real_mvp_results_{timestamp}.csv")
        df.to_csv(output_path, index=False)
        print(f"✅ Đã lưu kết quả thực tế vào: {output_path}")
        
except Exception as e:
    print(f"Xảy ra lỗi trong quá trình xử lý thực tế: {e}")

